<a href="https://colab.research.google.com/github/Carmuzqui/Teste-Maps-Cplex/blob/main/Tentativa_Python_Rotas_OD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Instalação das bibliotecas necessárias
!pip install geopandas shapely requests tqdm

import pandas as pd
import geopandas as gpd
from shapely.geometry import shape
import requests
import time
from tqdm import tqdm # Biblioteca para barra de progresso

print("Carregando os arquivos de dados...")
# 2. Carregar os dados
df_cargas = pd.read_csv('base_dados_Acucar_v08 (2).csv')
df_municipios = pd.read_csv('municipios.csv')

# Filtramos apenas as colunas que importam do arquivo de municípios
df_coords = df_municipios[['codigo_ibge', 'latitude', 'longitude', 'nome']]

# 3. Cruzar as coordenadas de ORIGEM
df_cargas = df_cargas.merge(
    df_coords, left_on='mun_origem', right_on='codigo_ibge', how='left'
)
df_cargas.rename(columns={'latitude': 'lat_origem', 'longitude': 'lon_origem', 'nome': 'nome_origem'}, inplace=True)
df_cargas.drop(columns=['codigo_ibge'], inplace=True)

# 4. Cruzar as coordenadas de DESTINO
df_cargas = df_cargas.merge(
    df_coords, left_on='mun_destino', right_on='codigo_ibge', how='left'
)
df_cargas.rename(columns={'latitude': 'lat_destino', 'longitude': 'lon_destino', 'nome': 'nome_destino'}, inplace=True)
df_cargas.drop(columns=['codigo_ibge'], inplace=True)

# Remove possíveis linhas que ficaram sem coordenada
df_cargas = df_cargas.dropna(subset=['lat_origem', 'lat_destino'])

# 5. Função que se comunica com o OSRM para gerar o traçado
def obter_rota_osrm(lon_origem, lat_origem, lon_destino, lat_destino):
    url = f"http://router.project-osrm.org/route/v1/driving/{lon_origem},{lat_origem};{lon_destino},{lat_destino}"
    params = {'overview': 'full', 'geometries': 'geojson'}

    try:
        resposta = requests.get(url, params=params)
        dados = resposta.json()
        if dados.get('code') == 'Ok':
            return shape(dados['routes'][0]['geometry'])
    except:
        pass
    return None

# 6. Roteamento
# ⚠️ ATENÇÃO: Aqui definimos para rodar apenas as 20 primeiras rotas para teste.
# Quando quiser rodar a planilha inteira, mude para: df_processar = df_cargas
df_processar = df_cargas

geometrias = []
rotas_validas = []

print(f"\nIniciando cálculo de {len(df_processar)} rotas na malha viária...")

# O 'tqdm' cria a barra de progresso visual
for index, row in tqdm(df_processar.iterrows(), total=len(df_processar)):
    time.sleep(1) # Pausa obrigatória para não derrubar o servidor público

    linha_rota = obter_rota_osrm(
        row['lon_origem'], row['lat_origem'],
        row['lon_destino'], row['lat_destino']
    )

    if linha_rota:
        geometrias.append(linha_rota)
        rotas_validas.append(row)

# 7. Salvar e Exportar
if rotas_validas:
    df_resultados = pd.DataFrame(rotas_validas)
    gdf_rotas = gpd.GeoDataFrame(df_resultados, geometry=geometrias, crs="EPSG:4326")

    # Exporta no formato moderno e leve para o ArcGIS
    nome_arquivo = "rotas_acucar_teste.gpkg"
    gdf_rotas.to_file(nome_arquivo, driver="GPKG")
    print(f"\n✅ Processo concluído! Arquivo '{nome_arquivo}' salvo com sucesso.")
    print("Atualize a aba lateral esquerda (Arquivos) para fazer o download.")
else:
    print("\n❌ Nenhuma rota válida foi calculada.")

Carregando os arquivos de dados...

Iniciando cálculo de 17433 rotas na malha viária...


100%|██████████| 17433/17433 [6:39:27<00:00,  1.37s/it]



✅ Processo concluído! Arquivo 'rotas_acucar_teste.gpkg' salvo com sucesso.
Atualize a aba lateral esquerda (Arquivos) para fazer o download.


In [2]:
!pip install geopandas shapely requests tqdm

import pandas as pd
import geopandas as gpd
from shapely.geometry import shape
import requests
import time
from tqdm import tqdm
import os # Biblioteca para manipular os nomes dos arquivos

# =========================================================================
# ⚙️ PAINEL DE CONFIGURAÇÃO
# Digite exatamente o nome do arquivo de cargas que você subiu no Colab:
ARQUIVO_ENTRADA = '/content/drive/MyDrive/Matrizes OD/base_dados_Adubos-e-fertilizantes_v09.csv'

# O Python cria os nomes de saída automaticamente com base no arquivo de entrada
nome_base = os.path.splitext(ARQUIVO_ENTRADA)[0]
ARQUIVO_SAIDA_FINAL = f"{nome_base}_com_rotas.gpkg"
# =========================================================================

print(f"1. Carregando os dados de origem e destino do arquivo: {ARQUIVO_ENTRADA}...")
df_cargas = pd.read_csv(ARQUIVO_ENTRADA)
df_municipios = pd.read_csv('/content/drive/MyDrive/Arquivos de configuracao/municipios.csv')

# Preparação das coordenadas
df_coords = df_municipios[['codigo_ibge', 'latitude', 'longitude', 'nome']]

# Cruzamento Origem
df_cargas = df_cargas.merge(df_coords, left_on='mun_origem', right_on='codigo_ibge', how='left')
df_cargas.rename(columns={'latitude': 'lat_origem', 'longitude': 'lon_origem', 'nome': 'nome_origem'}, inplace=True)
df_cargas.drop(columns=['codigo_ibge'], inplace=True)

# Cruzamento Destino
df_cargas = df_cargas.merge(df_coords, left_on='mun_destino', right_on='codigo_ibge', how='left')
df_cargas.rename(columns={'latitude': 'lat_destino', 'longitude': 'lon_destino', 'nome': 'nome_destino'}, inplace=True)
df_cargas.drop(columns=['codigo_ibge'], inplace=True)
df_cargas = df_cargas.dropna(subset=['lat_origem', 'lat_destino'])

print("2. Configurando roteamento otimizado...")
def obter_rota_osrm_dinamico(lon_origem, lat_origem, lon_destino, lat_destino, tentativas=3):
    url = f"http://router.project-osrm.org/route/v1/driving/{lon_origem},{lat_origem};{lon_destino},{lat_destino}"
    params = {'overview': 'full', 'geometries': 'geojson'}

    for tentativa in range(tentativas):
        try:
            resposta = requests.get(url, params=params)
            if resposta.status_code == 429: # Servidor bloqueou por velocidade
                time.sleep(2) # Espera 2 segundos de punição e tenta de novo
                continue

            dados = resposta.json()
            if dados.get('code') == 'Ok':
                return shape(dados['routes'][0]['geometry'])
            else:
                return None
        except Exception as e:
            time.sleep(1) # Falha de rede genérica
    return None

# =========================================================================
# ⚠️ ÁREA DE TESTE OU EXECUÇÃO TOTAL
# Para rodar a matriz completa, use: df_processar = df_cargas
# Para testar, use: df_processar = df_cargas.head(100)
df_processar = df_cargas
# =========================================================================

geometrias = []
rotas_validas = []
contador_salvamento = 0

print(f"\n3. Iniciando cálculo de {len(df_processar)} rotas na malha viária...")

for index, row in tqdm(df_processar.iterrows(), total=len(df_processar)):
    time.sleep(0.3) # Tempo otimizado

    linha_rota = obter_rota_osrm_dinamico(
        row['lon_origem'], row['lat_origem'],
        row['lon_destino'], row['lat_destino']
    )

    if linha_rota:
        geometrias.append(linha_rota)
        rotas_validas.append(row)
        contador_salvamento += 1

    # Sistema de Checkpoint de Segurança usa o nome base do arquivo agora
    if contador_salvamento == 2000:
        nome_backup = f"backup_{nome_base}_linha_{index}.gpkg"
        print(f"\n[Backup] Salvando progresso parcial em '{nome_backup}'...")
        df_parcial = pd.DataFrame(rotas_validas)
        gdf_parcial = gpd.GeoDataFrame(df_parcial, geometry=geometrias, crs="EPSG:4326")
        gdf_parcial.to_file(nome_backup, driver="GPKG")
        contador_salvamento = 0

print("\n4. Finalizando e exportando para o ArcGIS...")
if rotas_validas:
    df_resultados = pd.DataFrame(rotas_validas)
    gdf_rotas = gpd.GeoDataFrame(df_resultados, geometry=geometrias, crs="EPSG:4326")

    gdf_rotas.to_file(ARQUIVO_SAIDA_FINAL, driver="GPKG")
    print(f"✅ Processo concluído! Arquivo principal '{ARQUIVO_SAIDA_FINAL}' salvo com sucesso.")
else:
    print("❌ Nenhuma rota válida foi calculada.")

1. Carregando os dados de origem e destino do arquivo: /content/drive/MyDrive/Matrizes OD/base_dados_Adubos-e-fertilizantes_v09.csv...
2. Configurando roteamento otimizado...

3. Iniciando cálculo de 28688 rotas na malha viária...


  7%|▋         | 1999/28688 [23:27<6:18:44,  1.17it/s]


[Backup] Salvando progresso parcial em 'backup_/content/drive/MyDrive/Matrizes OD/base_dados_Adubos-e-fertilizantes_v09_linha_2699.gpkg'...


  7%|▋         | 1999/28688 [23:29<5:13:36,  1.42it/s]


DataSourceError: sqlite3_open(backup_/content/drive/MyDrive/Matrizes OD/base_dados_Adubos-e-fertilizantes_v09_linha_2699.gpkg) failed: unable to open database file